# Met Eyes Experiments

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

In [ ]:
import json
import requests

from os import listdir, path
from PIL import Image as PImage
from time import sleep

from utils import export_combined_jsons

DATA_DIR = "./data"
IMG_DIR = f"{DATA_DIR}/image"
JSON_DIR = f"{DATA_DIR}/json"

In [ ]:
MET_URL = "https://collectionapi.metmuseum.org/public/collection/v1"

SEARCH_DEPARTMENT_IDS = []
SEARCH_DEPARTMENTS = ["Robert Lehman", "Armor"]
SEARCH_MEDIUMS = ["Paintings", "Drawings"][:1]

### Get Department IDs

In [ ]:
dept_response = requests.get(f"{MET_URL}/departments")
dept_data = dept_response.json()["departments"]

dept_name2id = { d["displayName"] : d["departmentId"] for d in dept_data }

for sdpt in SEARCH_DEPARTMENTS:
  for dname,did in dept_name2id.items():
    if sdpt.lower() in dname.lower():
      SEARCH_DEPARTMENT_IDS.append(did)

### Get Object IDs

In [ ]:
obj_ids = []

for dpt_query in SEARCH_DEPARTMENT_IDS:
  for medium_query in SEARCH_MEDIUMS:
    collection_response = requests.get(f"{MET_URL}/search?medium={medium_query}&departmentId={dpt_query}&q=*")
    query_obj_ids = set(collection_response.json()["objectIDs"])
    obj_ids += list(query_obj_ids)

len(obj_ids)

### Get Object Metadata

In [ ]:
obj_fields = ["objectID", "objectName", "title", "primaryImage", "primaryImageSmall", "artistRole", "artistDisplayName"]
obj_files = sorted(f for f in listdir(f"{JSON_DIR}/objects") if f.endswith("json"))

for oid in obj_ids:
  obj_json_path = f"{JSON_DIR}/objects/{oid}.json"
  if f"{oid}.json" in obj_files:
    continue

  obj_response = requests.get(f"{MET_URL}/objects/{oid}")
  obj_data = obj_response.json()
  obj_filtered_data = { f: obj_data[f] for f in obj_fields }

  obj_json_path = f"{JSON_DIR}/objects/{oid}.json"
  with open(obj_json_path, "w") as ofp:
    json.dump(obj_filtered_data, ofp)
  sleep(0.333)

### Export Combined Object Metadata

In [ ]:
export_combined_jsons(f"{JSON_DIR}/objects", JSON_DIR, "objects")

### Get Images

In [ ]:
with open(f"{JSON_DIR}/objects.json", "r") as ifp:
  obj_data = json.load(ifp)["objects"]

for obj in obj_data:
  img_url = obj["primaryImage"]
  if not (img_url and len(img_url) > 0):
    continue

  oid = obj["objectID"]
  img_hd_path = f"{IMG_DIR}/hd/{oid}.jpg"
  img_900_path = img_hd_path.replace("/hd/", "/900/")
  img_500_path = img_hd_path.replace("/hd/", "/500/")
  if path.isfile(img_hd_path) and path.isfile(img_900_path) and path.isfile(img_500_path):
    continue

  img_response = requests.get(img_url, stream=True)
  img = PImage.open(img_response.raw)

  img.thumbnail((2048, 2048))
  if not path.isfile(img_hd_path):
    img.save(img_hd_path)

  img.thumbnail((900, 900))
  if not path.isfile(img_900_path):
    img.save(img_900_path)

  img.thumbnail((500, 500))
  if not path.isfile(img_500_path):
    img.save(img_500_path)

  sleep(0.333)

## Analyze Faces

### Face Detection

In [ ]:
!git clone https://github.com/acervos-digitais/met-faces-data.git data
!pip install ultralytics

In [ ]:
import json
import numpy as np

from os import listdir
from PIL import Image as PImage, ImageDraw as PImageDraw

from huggingface_hub import hf_hub_download
from torch import no_grad, Tensor
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, pipeline
from ultralytics import YOLO

from utils import export_combined_jsons, pxs_to_pcts, draw_boxes

DATA_DIR = "./data"
IMG_DIR = f"{DATA_DIR}/image"
JSON_DIR = f"{DATA_DIR}/json"

In [ ]:
img_ids = sorted(int(fn.replace(".jpg", "")) for fn in listdir(f"{IMG_DIR}/hd") if fn.endswith(".jpg"))

with open(f"{JSON_DIR}/objects.json", "r") as ifp:
  obj_data = json.load(ifp)["objects"]
  id2obj = { obj["objectID"] : obj for obj in obj_data }

### Detect Faces ([YOLO11](https://huggingface.co/AdamCodd/YOLOv11n-face-detection))

In [ ]:
yolo_model_path = hf_hub_download(repo_id="AdamCodd/YOLOv11n-face-detection", filename="model.pt")
face_detector = YOLO(yolo_model_path)

In [ ]:
for oid in img_ids:
  obj = id2obj[oid]

  if "faces" in obj and "yolo" in obj["faces"]:
    continue

  img = PImage.open(f"{IMG_DIR}/500/{oid}.jpg")
  iw,ih = img.size
  nh = 256
  nw = int(nh * iw // ih)
  nimg = img.resize((nw, nh))

  faces = face_detector.predict(nimg, verbose=False, device="cuda")
  if len(faces) < 1 or len(faces[0]) < 1:
    continue

  faces_xyxyn = faces[0].boxes.xyxyn.cpu().numpy().astype(np.float64)
  # faces_xywhn = faces[0].boxes.xywhn.cpu().numpy().astype(np.float64)

  if "faces" not in obj:
    obj["faces"] = {}

  obj["faces"]["yolo"] = {
      "count": len(faces_xyxyn),
      "xyxyn": faces_xyxyn.round(4).tolist(),
      # "xywhn": faces_xywhn.round(4).tolist(),
  }

  with open(f"{JSON_DIR}/objects/{oid}.json", "w") as ofp:
    json.dump(obj, ofp)

### Detect Faces (Zero-Shot)

In [ ]:
MODEL_NAME = "IDEA-Research/grounding-dino-base"

zs_processor = AutoProcessor.from_pretrained(MODEL_NAME)
zs_model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_NAME).to("cuda")

labels = ["face"]

In [ ]:
for oid in img_ids:
  obj = id2obj[oid]

  if "faces" in obj and "dino" in obj["faces"]:
    continue

  img = PImage.open(f"{IMG_DIR}/500/{oid}.jpg")
  iw,ih = img.size

  with no_grad():
    input = zs_processor(text=labels, images=img, return_tensors="pt").to("cuda")
    output = zs_model(**input)

  res = zs_processor.post_process_grounded_object_detection(outputs=output, target_sizes=[Tensor([ih, iw])], threshold=0.33)

  if len(res[0]["boxes"]) < 1:
    continue

  pct_boxes = pxs_to_pcts(res[0]["boxes"].cpu(), iw, ih)
  faces_xyxyn = pct_boxes[1].astype(np.float64)
  # faces_xywhn = pct_boxes[0].astype(np.float64)

  if "faces" not in obj:
    obj["faces"] = {}

  obj["faces"]["dino"] = {
      "count": len(res[0]["boxes"]),
      "xyxyn": faces_xyxyn.round(4).tolist(),
      # "xywhn": faces_xywhn.round(4).tolist(),
  }

  with open(f"{JSON_DIR}/objects/{oid}.json", "w") as ofp:
    json.dump(obj, ofp)

In [ ]:
export_combined_jsons(f"{JSON_DIR}/objects", JSON_DIR, "faces")

In [ ]:
# TODO: Landmark Detection
# TODO: OpenCV
# TODO: https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker/index
# TODO: https://huggingface.co/kartiknarayan/facexformer
# TODO: https://huggingface.co/qualcomm/Facial-Landmark-Detection